# Clustering analysis

K-means on the pipeline features, to see which groups of items and stores sell more. The raw data is at https://www.kaggle.com/c/m5-forecasting-accuracy/data.

In [ ]:
# run from the project folder so data/ paths work and src/ is importable
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

In [2]:
import plotly.express as px
from sklearn.cluster import KMeans

from m5_forecasting.clustering import ClusterScoreTracker
from m5_forecasting.features import FeaturePipeline

## Build features

In [3]:
pipeline = FeaturePipeline(path="data/raw/", verbose=False)
sales = pipeline.build_feature_set()

## Preprocessing

Drop the id columns and missing values, then sample 1M rows.

In [5]:
excluded_columns = [
    "id", "item_id", "dept_id", "store_id", "cat_id", "state_id",
    "d", "sales", "date", "wm_yr_wk",
]
feature_columns = [column for column in sales.columns if column not in excluded_columns]

sales = sales.dropna()
sales = sales.sample(1_000_000)

## Try 2 to 14 clusters

In [ ]:
tracker = ClusterScoreTracker()

for n_clusters in range(2, 15):
    kmeans = KMeans(n_clusters=n_clusters, max_iter=2000, tol=0.00001)
    kmeans.fit(sales[feature_columns].values)
    tracker.add_result(kmeans, sales[feature_columns].values, n_clusters)

tracker.plot()

In [ ]:
scores = tracker.scores.copy()
scores["score"] = scores["score"] / 1_000_000
scores

## K-means with 4 clusters

In [ ]:
kmeans = KMeans(n_clusters=4, max_iter=2000, tol=0.00001)
sales["cluster"] = kmeans.fit_predict(sales[feature_columns].values)

## Sales and price variety by cluster

In [ ]:
cluster_means = sales.groupby("cluster")[["sales", "item_nunique_price"]].mean().reset_index()
cluster_means = cluster_means.rename(
    columns={"cluster": "Cluster", "sales": "Sales", "item_nunique_price": "Unique Items"}
)
cluster_means["Cluster"] = "Cluster " + cluster_means["Cluster"].astype(str)

figure = px.bar(cluster_means, x="Cluster", y="Sales", color="Unique Items")
figure.update_layout(autosize=False, width=600, height=600)
figure.show()

## Save data

Saved for the R scripts.

In [ ]:
sales.to_csv("data/refined/clustering_data.csv")